In [1]:
import torch
from torchvision.utils import make_grid
import imageio
import torch.nn as nn
from torch.nn import functional as F
import numpy as np

In [2]:
# Upsample Block using ConvTranspose2d for Generator
class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UpsampleBlock, self).__init__()
        self.upblock = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
            )
        self.shortcut = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        out = self.upblock(x)
        shortcut = self.shortcut(x)
        return out + shortcut

# Downsample Block using Conv2d for Discriminator
class DownsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DownsampleBlock, self).__init__()
        self.downblock = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 2, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
    
    def forward(self, x):
        out = self.downblock(x)
        shortcut = self.shortcut(x)
        return out + shortcut

# Regular ConvBlock (Residual) 
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.convblock = nn.Sequential( 
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        
    def forward(self, x):
        out = self.convblock(x)
        shortcut = self.shortcut(x)
        return out + shortcut

# Generator Architecture
class Generator(nn.Module):
    def __init__(self, img_size, latent_dim=100):
        super(Generator, self).__init__()

        self.img_size = img_size
        self.latent_dim = latent_dim

        self.latent_to_features = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.GELU(),
            nn.Linear(256, 512),
            nn.GELU(),
            nn.Linear(512, 512 * 32 * 32)
        )

        self.features_to_latent = nn.Sequential(
            ConvBlock(512, 256),
            UpsampleBlock(256, 128),
            ConvBlock(128, 64),
            UpsampleBlock(64, 32),
            ConvBlock(32, 3),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.latent_to_features(x)
        x = x.view(-1, 512, 32, 32)
        x = self.features_to_latent(x)
        return x
    
    def sample_latent(self, num_samples, device="cpu"):
        return torch.rand((num_samples, self.latent_dim)).to(device)

# Discriminator Architecture    
class Discriminator(nn.Module):
    def __init__(self, in_channels):
        super(Discriminator, self).__init__()

        self.in_channels = in_channels

        self.cnn = nn.Sequential(
            DownsampleBlock(self.in_channels, 1),
            ConvBlock(1, 1),
            ConvBlock(1, 1),
            DownsampleBlock(1, 1),
            ConvBlock(1, 1),
            ConvBlock(1, 1)
        )

        self.prob = nn.Sequential(
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.cnn(x)
        x = x.flatten()
        x = x.view(-1, 1024)
        x = self.prob(x)
        return x


In [ ]:
class Trainer():
    def __init__(self, generator, discriminator, g_optim, d_optim, gp_w=10, critic_iters=5, logs=50, cuda=False):
        # Generator
        self.G = generator
        self.g_opt = g_optim
        
        # Discriminator
        self.D = discriminator
        self.d_opt = d_optim

        self.losses = {"G": [], "D": [], "GP": [], "gradient_norm": []}
        self.num_steps = 0
        self.cuda = cuda
        self.gp_w = gp_w
        self.critic_iters = critic_iters
        self.logs = logs 

        if self.cuda:
            self.G = self.G.cuda()
            self.D = self.D.cuda()
    
    def __gradient_penalty(self, real, generated):
        batch_size = real.size(0)
        
        # Calculate interpolation
        alpha = torch.rand(batch_size, 1, 1, 1)
        alpha = alpha.expand_as(real)
        if self.cuda:
            alpha = alpha.cuda()
        interpolated = alpha * real.data + (1 - alpha) * generated.data
        interpolated = torch.autograd.Variable(interpolated, requires_grad=True)
        if self.cuda:
            interpolated = interpolated.cuda()
        
        # Calculate Probability of Interpolated Examples
        prob_inter = self.D(interpolated)
        gradients = torch.autograd.grad(outputs=prob_inter, inputs=interpolated, 
                               grad_outputs=torch.ones(prob_inter.size()).cuda() if self.cuda
                               else torch.ones(prob_inter.size()),
                               create_graph=True, retain_graph=True)[0]

        gradients = gradients.view(batch_size, -1)
        self.losses["gradient_norm"].append(gradients.norm(2, dim=1).mean().item())

        gradients_norm = torch.sqrt(torch.sum(gradients**2, dim=1) + 1e-12)

        return self.gp_w * ((gradients_norm - 1) ** 2).mean()

    def __critic_train_iter(self, data):
        # Generate Fake Data
        batch_size = data.size(0)
        generated_data = self.G(self.G.sample_latent(batch_size, "cuda" if torch.cuda.is_available() else "cpu"))

        # Calculate Discriminator Outputs for Real and Fake Data
        data = torch.autograd.Variable(data)
        if self.cuda:
            data = data.cuda()
        d_real = self.D(data)
        d_fake = self.D(generated_data)

        # Calculate Gradient Penalty
        gradient_penalty = self.__gradient_penalty(data, generated_data)
        self.losses["GP"].append(gradient_penalty.item())
        
        # Update Discriminator
        self.d_opt.zero_grad()
        d_loss = d_fake.mean() - d_real.mean() + gradient_penalty
        d_loss.backward()

        self.d_opt.step()

        # Record Loss
        self.losses["D"].append(d_loss.item())

    def __generator_train_iter(self, data):
        # Generate Fake Data
        batch_size = data.size(0)
        generated_data = self.G(self.G.sample_latent(batch_size, "cuda" if torch.cuda.is_available() else "cpu"))
        
        # Calculate D Loss and Optimize
        self.g_opt.zero_grad()
        d_generated = self.D(generated_data)
        g_loss = -d_generated.mean()
        g_loss.backward()
        self.g_opt.step()

        self.losses["G"].append(g_loss.item())

    def __train_ep(self, data_loader):
        for i, data in enumerate(data_loader):
            self.num_steps += 1
            self.__critic_train_iter(data)

            # Only Update Generator after [critic_iters] iterations
            if self.num_steps % self.critic_iters == 0:
                self.__generator_train_iter(data)

            if i % self.logs == 0:
                print("Iteration {}".format(i+1))
                print("D: {}".format(self.losses["D"][-1]))
                print("GP: {}".format(self.losses["GP"][-1]))
                print("Gradient Norm: {}".format(self.losses["gradient_norm"][-1]))

                if self.num_steps > self.critic_iters:
                    print("G: {}".format(self.losses["G"][-1]))

    def train(self, data_loader, eps, save=True):
        if save:
            fixed_latents = torch.autograd.Variable(self.G.sample_latent(64))
            if self.cuda:
                fixed_latents = fixed_latents.cuda()
            training_progress_imgs = []

        for ep in range(eps):
            print("\nEpoch {}".format(ep + 1))
            self.__train_ep(data_loader)

            if save:
                img_grid = make_grid(self.G(fixed_latents).cpu().data)
                img_grid = np.transpose(img_grid.numpy(), (1, 2, 0))
                training_progress_imgs.append(img_grid)
        
        if save:
            imageio.mimsave("./training_{}_epochs.gif".format(eps), training_progress_imgs)

    def sample_generator(self, num_samples):
        latent_samples = torch.autograd.Variable(self.G.sample_latent(num_samples))
        if self.cuda:
            latent_samples = latent_samples.cuda()
        generated_data = self.G(latent_samples)
        return generated_data

    def sample(self, num_samples):
        generated_data = self.sample_generator(num_samples)

        return generated_data.data.cpu().numpy()[:, 0, :, :]    

In [4]:
data = torch.load("./data/train.pt", weights_only=False)
train_data = torch.utils.data.DataLoader(data["data"].to(torch.float32), batch_size=64, shuffle=True)

In [5]:
d_model = Discriminator(3)
g_model = Generator((3, 128, 128), 100)

lr = 1e-4
betas = (.9, .99)
eps = 200

d_optim = torch.optim.AdamW(d_model.parameters(), lr=lr, betas=betas)
g_optim = torch.optim.AdamW(g_model.parameters(), lr=lr, betas=betas)


trainer = Trainer(generator=g_model, discriminator=d_model, g_optim=g_optim, d_optim=d_optim, cuda=torch.cuda.is_available())

In [ ]:
trainer.train(train_data, eps, save=True)


Epoch 1
Iteration 1
D: 9.994478225708008
GP: 9.972819328308105
Gradient Norm: 0.0013601930113509297
Iteration 51
D: 9.102811813354492
GP: 9.907392501831055
Gradient Norm: 0.004643997177481651
G: -0.03967069834470749
Iteration 101
D: 8.80219554901123
GP: 9.668469429016113
Gradient Norm: 0.016764257103204727
G: -0.0009547509253025055

Epoch 2
Iteration 1
D: 8.763564109802246
GP: 9.627352714538574
Gradient Norm: 0.018879126757383347
G: -0.0009547509253025055
Iteration 51
D: 8.175162315368652
GP: 8.978119850158691
Gradient Norm: 0.05346379801630974
G: -0.0006642636144533753
Iteration 101
D: 7.811197280883789
GP: 8.694748878479004
Gradient Norm: 0.07057991623878479
G: -3.3575171983102337e-05

Epoch 3
Iteration 1
D: 7.580993175506592
GP: 8.497868537902832
Gradient Norm: 0.08132204413414001
G: -0.00010160509555134922
Iteration 51
D: 8.639151573181152
GP: 8.613630294799805
Gradient Norm: 0.07442283630371094
G: -0.0006559875328093767
Iteration 101
D: 7.56614875793457
GP: 8.373817443847656
Grad